<div style="background:linear-gradient(135deg,#0F3D6E 0%,#2176AE 100%);padding:40px 36px 32px 36px;border-radius:10px;margin-bottom:8px;">
  <p style="color:#C8DEF5;font-size:13px;margin:0 0 6px 0;letter-spacing:2px;">CURSO 8 · MÓDULO 1 · CLASE 3</p>
  <h1 style="color:white;font-size:36px;margin:0 0 10px 0;font-weight:700;">Formulación OLS, Supuestos Gauss-Markov y Propiedad BLUE</h1>
  <p style="color:#C8DEF5;font-size:16px;margin:0 0 24px 0;font-style:italic;">Por qué β̂ es el mejor estimador posible — y cuándo deja de serlo</p>
  <hr style="border-color:#5BA4CF;margin:0 0 20px 0;">
  <p style="color:#EAF2FB;font-size:13px;margin:0;">📌 <strong>Docente:</strong> Josef Rodriguez &nbsp;·&nbsp; <strong>Nivel:</strong> Avanzado &nbsp;·&nbsp; <strong>Duración:</strong> 2 horas</p>
</div>

## Objetivos

| # | Al terminar podés |
|---|-------------------|
| 1 | Enunciar los 5 supuestos de Gauss-Markov y saber cuál viola qué |
| 2 | Demostrar empíricamente que β̂ OLS es insesgado |
| 3 | Entender qué significa BLUE y por qué OLS lo logra |
| 4 | Estimar σ² con el estimador insesgado s² = SSE/(n−p) |
| 5 | Detectar violaciones a los supuestos con gráficos de diagnóstico |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.dpi':110,'font.size':11,
                     'axes.spines.top':False,'axes.spines.right':False})
SEED = 42; np.random.seed(SEED)
print('✅ Setup listo')

---
## 1. El modelo lineal y sus supuestos

**Modelo:** y = Xβ + ε

Los **5 supuestos de Gauss-Markov** que hacen que β̂ OLS sea BLUE:

| # | Supuesto | Notación | Si se viola |
|---|----------|----------|-------------|
| GM1 | Linealidad en parámetros | y = Xβ + ε | Modelo mal especificado |
| GM2 | X no estocástica o independiente de ε | E[ε\|X] = 0 | Endogeneidad |
| GM3 | Esperanza cero del error | E[ε] = 0 | Intercepto mal definido |
| GM4 | Homocedasticidad | Var(ε) = σ²I | Heterocedasticidad → WLS |
| GM5 | No autocorrelación | Cov(εᵢ, εⱼ) = 0 ∀i≠j | Series de tiempo → GLS |

In [ ]:
# Simular datos que CUMPLEN todos los supuestos GM
np.random.seed(SEED)
n, p = 300, 4
beta_true = np.array([5.0, 2.0, -1.5, 3.0])
sigma_true = 1.2

X_raw = np.random.randn(n, p-1)
X = np.column_stack([np.ones(n), StandardScaler().fit_transform(X_raw)])

# ε ~ N(0, σ²I) — homocedastico, no correlacionado
epsilon = np.random.normal(0, sigma_true, n)
y = X @ beta_true + epsilon

print(f'Dataset: {n} obs × {p} parámetros')
print(f'β real:  {beta_true}')
print(f'σ real:  {sigma_true}')

---
## 2. Insesgadez de β̂

**Prueba algebraica:** E[β̂] = E[(XᵀX)⁻¹Xᵀy] = β

Porque y = Xβ + ε y E[ε] = 0:

E[β̂] = (XᵀX)⁻¹Xᵀ E[y] = (XᵀX)⁻¹Xᵀ Xβ = β

Lo verificamos empíricamente con simulaciones.

In [ ]:
# Verificar insesgadez empíricamente
np.random.seed(SEED)
n_sim = 5000
X_fijo = X.copy()   # diseño fijo
XtX_inv = np.linalg.inv(X_fijo.T @ X_fijo)

beta_hats = np.zeros((n_sim, p))
for s in range(n_sim):
    eps_s = np.random.normal(0, sigma_true, n)
    y_s   = X_fijo @ beta_true + eps_s
    beta_hats[s], *_ = np.linalg.lstsq(X_fijo, y_s, rcond=None)

print('Verificación de insesgadez (E[β̂] = β):')
print(f'{"Parámetro":15s} {"β real":>8s} {"E[β̂] empír":>12s} {"Sesgo":>10s}')
print('─'*50)
for i, (real, emp) in enumerate(zip(beta_true, beta_hats.mean(axis=0))):
    print(f'β{i} {"":12s} {real:>8.3f} {emp:>12.4f} {abs(emp-real):>10.5f}')

In [ ]:
# Visualizar distribución centrada en β_real
fig, axes = plt.subplots(1, p, figsize=(14, 3.5))
for i, ax in enumerate(axes):
    ax.hist(beta_hats[:, i], bins=60, density=True, color='#2176AE',
            edgecolor='white', alpha=0.8)
    ax.axvline(beta_true[i], color='#C0392B', lw=2, label=f'β_real={beta_true[i]}')
    ax.axvline(beta_hats[:,i].mean(), color='#0F3D6E', lw=1.5,
               linestyle='--', label='E[β̂]')
    ax.set_title(f'β{i}', fontweight='bold')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.2)
plt.suptitle('β̂ OLS es INSESGADO: la distribución está centrada en β_real (5000 sims)',
             fontweight='bold')
plt.tight_layout(); plt.show()

---
## 3. Estimación de σ² con s²

σ² (varianza del error) es desconocida. El estimador insesgado es:

$$s^2 = \frac{SSE}{n-p} = \frac{\mathbf{e}^\top\mathbf{e}}{n-p}$$

**¿Por qué n−p y no n?** Porque estimamos p parámetros, perdemos p grados de libertad.

In [ ]:
# Demostrar que s² = SSE/(n-p) es insesgado para σ²
np.random.seed(SEED)
n_sim = 5000
sigma2_true = sigma_true ** 2
s2_estimates_np = []
s2_estimates_n  = []

for s in range(n_sim):
    eps_s = np.random.normal(0, sigma_true, n)
    y_s   = X_fijo @ beta_true + eps_s
    b_s, *_ = np.linalg.lstsq(X_fijo, y_s, rcond=None)
    e_s   = y_s - X_fijo @ b_s
    SSE_s = e_s @ e_s
    s2_estimates_np.append(SSE_s / (n - p))   # insesgado
    s2_estimates_n.append(SSE_s / n)           # sesgado

s2_np = np.array(s2_estimates_np)
s2_n  = np.array(s2_estimates_n)

print(f'σ² real:            {sigma2_true:.4f}')
print(f'E[SSE/(n-p)]:       {s2_np.mean():.4f}  ← insesgado ✓')
print(f'E[SSE/n]:           {s2_n.mean():.4f}  ← sesgado hacia abajo')
print(f'\nSesgo con n:        {s2_n.mean() - sigma2_true:.4f}  (subestima σ²)')

In [ ]:
# Estimar σ² en nuestro dataset
beta_hat, *_ = np.linalg.lstsq(X, y, rcond=None)
e   = y - X @ beta_hat
SSE = e @ e
s2  = SSE / (n - p)
s   = np.sqrt(s2)

print(f'σ real:             {sigma_true:.4f}')
print(f's = √(SSE/(n-p)):   {s:.4f}  ← estimado')
print(f'\nSSE = {SSE:.4f}')
print(f'n-p = {n-p} grados de libertad residuales')

---
## 4. La Propiedad BLUE

**BLUE** = Best Linear Unbiased Estimator

El **Teorema de Gauss-Markov** dice: bajo GM1–GM5, β̂ OLS tiene la **menor varianza** entre todos los estimadores lineales insesgados.

Intuición: cualquier otro estimador lineal β̃ = Cy con E[β̃] = β tendrá Var(β̃) ≥ Var(β̂).

In [ ]:
# Demostrar BLUE: comparar varianza de OLS con un estimador alternativo
np.random.seed(SEED)
n_sim2 = 3000
n2, p2 = 100, 3
beta2  = np.array([2.0, 1.5, -1.0])
sig2   = 1.0

X2 = np.column_stack([np.ones(n2), np.random.randn(n2, p2-1)])
XtX2_inv = np.linalg.inv(X2.T @ X2)

# Estimador OLS (β̂)
betas_ols = np.zeros((n_sim2, p2))
# Estimador alternativo: promedio simple de columnas
# β̃ = (X*ᵀX*) donde X* = X + ruido (perturbación del diseño)
betas_alt = np.zeros((n_sim2, p2))

for s in range(n_sim2):
    eps = np.random.normal(0, sig2, n2)
    y2  = X2 @ beta2 + eps

    # OLS
    betas_ols[s], *_ = np.linalg.lstsq(X2, y2, rcond=None)

    # Alternativo: usar solo las primeras 50 observaciones (pérdida de eficiencia)
    X2_sub = X2[:50]
    y2_sub = y2[:50]
    betas_alt[s], *_ = np.linalg.lstsq(X2_sub, y2_sub, rcond=None)

print('Varianzas de β̂ (menor = mejor):')
print(f'{"Param":8s} {"Var OLS":>12s} {"Var alternativo":>16s} {"Ratio":>8s}')
print('─'*46)
for i in range(p2):
    v_ols = betas_ols[:,i].var()
    v_alt = betas_alt[:,i].var()
    print(f'β{i}  {"":5s} {v_ols:>12.4f} {v_alt:>16.4f} {v_alt/v_ols:>8.2f}x')
print('\n→ OLS siempre tiene menor varianza (eficiencia ← propiedad BLUE)')

---
## 5. Diagnóstico de supuestos con gráficos

Los 4 gráficos estándar de diagnóstico OLS:

In [ ]:
# Datos que CUMPLEN supuestos (caso bueno)
np.random.seed(SEED)
n_d, p_d = 300, 4
beta_d = np.array([3.0, 1.5, -2.0, 1.0])
X_d = np.column_stack([np.ones(n_d), StandardScaler().fit_transform(np.random.randn(n_d, p_d-1))])
y_d = X_d @ beta_d + np.random.normal(0, 1.5, n_d)
beta_d_hat, *_ = np.linalg.lstsq(X_d, y_d, rcond=None)
y_d_hat = X_d @ beta_d_hat
e_d     = y_d - y_d_hat
n_d_obs, p_d_cols = X_d.shape
s2_d    = (e_d @ e_d) / (n_d_obs - p_d_cols)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1. Residuos vs Ajustados
axes[0,0].scatter(y_d_hat, e_d, alpha=0.4, s=14, color='#2176AE', edgecolors='none')
axes[0,0].axhline(0, color='#C0392B', lw=1.5, linestyle='--')
axes[0,0].set_xlabel('Valores ajustados ŷ')
axes[0,0].set_ylabel('Residuos e')
axes[0,0].set_title('1. Residuos vs Ajustados\n(sin patrón → homocedasticidad ✓)', fontweight='bold')
axes[0,0].grid(True, alpha=0.2)

# 2. Q-Q Plot de residuos estandarizados
e_std = e_d / np.sqrt(s2_d)
stats.probplot(e_std, dist='norm', plot=axes[0,1])
axes[0,1].set_title('2. Q-Q Plot de residuos\n(puntos sobre la línea → normalidad ✓)', fontweight='bold')
axes[0,1].lines[0].set(color='#2176AE', alpha=0.5, markersize=4)
axes[0,1].lines[1].set(color='#C0392B', lw=2)
axes[0,1].grid(True, alpha=0.2)

# 3. Scale-Location (√|e| vs ŷ)
axes[1,0].scatter(y_d_hat, np.sqrt(np.abs(e_std)), alpha=0.4, s=14,
                  color='#5BA4CF', edgecolors='none')
axes[1,0].set_xlabel('Valores ajustados ŷ')
axes[1,0].set_ylabel('√|eᵢ estandarizado|')
axes[1,0].set_title('3. Scale-Location\n(banda plana → varianza constante ✓)', fontweight='bold')
axes[1,0].grid(True, alpha=0.2)

# 4. Residuos vs orden (autocorrelación)
axes[1,1].plot(range(len(e_d)), e_d, '.', alpha=0.4, markersize=4, color='#2176AE')
axes[1,1].axhline(0, color='#C0392B', lw=1.5, linestyle='--')
axes[1,1].set_xlabel('Índice de observación')
axes[1,1].set_ylabel('Residuo e')
axes[1,1].set_title('4. Residuos vs Orden\n(sin tendencia → no autocorrelación ✓)', fontweight='bold')
axes[1,1].grid(True, alpha=0.2)

plt.suptitle('Panel de diagnóstico OLS — Caso que CUMPLE los supuestos GM',
             fontweight='bold', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Datos que VIOLAN homocedasticidad
np.random.seed(SEED)
x_het = np.random.uniform(1, 10, 300)
eps_het = np.random.normal(0, x_het * 0.5)   # varianza crece con x
y_het = 2 + 1.5 * x_het + eps_het
X_het = np.column_stack([np.ones(300), x_het])
b_het, *_ = np.linalg.lstsq(X_het, y_het, rcond=None)
e_het = y_het - X_het @ b_het

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(x_het, y_het, alpha=0.4, s=12, color='#2176AE', edgecolors='none')
axes[0].plot(x_het, X_het @ b_het, color='#C0392B', lw=2, label='OLS')
axes[0].set_title('Datos heterocedasticos — dispersión crece con x', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.2)

axes[1].scatter(X_het @ b_het, e_het, alpha=0.4, s=12, color='#C0392B', edgecolors='none')
axes[1].axhline(0, color='gray', lw=1, linestyle='--')
axes[1].set_xlabel('ŷ'); axes[1].set_ylabel('Residuos')
axes[1].set_title('Residuos vs Ajustados — patrón de embudo\n→ Heterocedasticidad ✗  usar WLS', fontweight='bold')
axes[1].grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

---
## 6. Aplicación: Verificar supuestos en datos de negocio

Pipeline completo con diagnóstico antes de interpretar resultados.

In [ ]:
# Dataset: ventas de tiendas retail
np.random.seed(SEED)
n_r = 250
precio       = np.random.uniform(10, 100, n_r)
descuento    = np.random.uniform(0, 0.5, n_r)
trafico      = np.random.gamma(5, 200, n_r)
temporada    = np.random.choice([0, 1], n_r, p=[0.6, 0.4]).astype(float)

# Ventas con errores homocedásticos
ventas = (5000
          - 30  * precio
          + 2000 * descuento
          + 1.2  * trafico
          + 800  * temporada
          + np.random.normal(0, 300, n_r))

# Construir X y estimar
feats_r = ['precio','descuento','trafico','temporada']
X_r_raw = np.column_stack([precio, descuento, trafico, temporada])
X_r     = np.column_stack([np.ones(n_r), StandardScaler().fit_transform(X_r_raw)])
b_r, *_ = np.linalg.lstsq(X_r, ventas, rcond=None)
yh_r    = X_r @ b_r
e_r     = ventas - yh_r
s2_r    = (e_r @ e_r) / (n_r - X_r.shape[1])

R2_r = r2_score(ventas, yh_r)
print(f'R² = {R2_r:.4f}')
print(f's  = {np.sqrt(s2_r):.2f} USD  (error estándar residual)')
print(f'\nCoeficientes (variables estandarizadas):')
for nm, b in zip(['intercepto']+feats_r, b_r):
    print(f'  {nm:14s}: {b:+.2f}')

In [ ]:
# Test de Shapiro-Wilk sobre residuos
_, p_sw = stats.shapiro(e_r[:250])
print(f'Shapiro-Wilk: p={p_sw:.4f} → residuos normales: {p_sw > 0.05}')

# Test de Durbin-Watson (autocorrelación)
dw = np.sum(np.diff(e_r)**2) / (e_r @ e_r)
print(f'Durbin-Watson: {dw:.4f}  (ideal ≈ 2.0, > 1.5 y < 2.5 → OK)')

# Breusch-Pagan manual (homocedasticidad)
e2 = e_r ** 2
b_bp, *_ = np.linalg.lstsq(X_r, e2, rcond=None)
e2_hat = X_r @ b_bp
SS_reg = ((e2_hat - e2.mean())**2).sum()
SS_tot = ((e2 - e2.mean())**2).sum()
R2_bp  = SS_reg / SS_tot
LM_bp  = n_r * R2_bp
p_bp   = 1 - stats.chi2.cdf(LM_bp, df=X_r.shape[1]-1)
print(f'Breusch-Pagan: LM={LM_bp:.2f}, p={p_bp:.4f} → homocedasticidad: {p_bp > 0.05}')

---
## Conclusiones

<div style="background:#EAF2FB;border-left:5px solid #2176AE;padding:20px 24px;border-radius:0 8px 8px 0;">

**01 · BLUE requiere GM1–GM5**  
Cada supuesto tiene su violación típica y su solución: heterocedasticidad → WLS, autocorrelación → GLS, endogeneidad → IV.

**02 · s² = SSE/(n−p) es el estimador correcto de σ²**  
Dividir por n en lugar de n−p subestima la varianza — error crítico en inferencia.

**03 · Los 4 gráficos de diagnóstico son obligatorios**  
Antes de interpretar cualquier coeficiente, verificar que los residuos no muestren patrones.

</div>

---
<div style="background:#0F3D6E;color:white;padding:20px 24px;border-radius:8px;">
<strong>Próxima clase — Lunes</strong><br>
R² y R² ajustado · Prueba F general · t-tests · Intervalos de confianza para β̂<br>
<em>Docente: Josef Rodriguez · Curso 8 · Modelos Estadísticos</em>
</div>